# 🚀 MIT Masterclass — Lecture 3: JAX Field Implementation
## *From Tokens to Trained Weights — Everything From Scratch in JAX*

---

> **"The most important paper of the last decade was not about attention. It was about data."**  
> — Prof. Edmund R. Whitmore, MIT CSAIL

---

### 🛰️ Mission Brief — Reid Wiseman Level

You are an ML systems engineer at a frontier lab. You've been handed Lecture 3 notes and told:

> *"We don't care that you can recite the theory. Prove you can build it. No libraries. No PyTorch. No HuggingFace. JAX only. From tokeniser to trained weights. You have the concepts — now make them run."*

This notebook is **not a tutorial**. It is a **mission**. Each section has:
- Working scaffold code in JAX
- **`# YOUR CODE HERE`** gaps that you must fill to make the cell pass
- A **CHECKPOINT** assertion that will fail loudly until your implementation is correct
- A **MISSION DEBRIEF** challenge that pushes beyond the lecture material

**Stack:** JAX (functional, JIT, vmap, grad). No PyTorch. No TF. No Flax unless noted.

---

### Environment Setup

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install jax jaxlib optax matplotlib numpy

import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, random, lax
import optax
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, NamedTuple
import re, time

# Reproducibility key — all randomness flows from here
MASTER_KEY = random.PRNGKey(42)

print(f"JAX version : {jax.__version__}")
print(f"Devices     : {jax.devices()}")
print(f"Backend     : {jax.default_backend()}")
print("\n✅ Environment ready. Mission clock started.")

---

## MISSION 1 — Tokenisation: Build BPE From Scratch

**Lecture 3, Sections 1–2.** No tokeniser libraries. Pure Python + NumPy.

### Background
BPE was a 1994 compression algorithm repurposed in 2015 for NLP. The algorithm:
1. Start with a character-level vocabulary
2. Count all adjacent token pairs in the corpus
3. Merge the most frequent pair into a new token
4. Repeat until target vocabulary size is reached

The vocabulary is then used to encode/decode text by greedily applying merges in learned order.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 1-A: Count Pair Frequencies
# ─────────────────────────────────────────────────────────────────────────────

# The corpus below mirrors Prof. Whitmore's worked example (Sec 2.2)
CORPUS = [
    "low", "low", "low", "low",
    "lower", "lower",
    "newest", "newest", "newest",
    "widest", "widest",
]

def word_to_chars(word: str) -> Tuple[str, ...]:
    """Split word into characters with end-of-word marker.
    'low' → ('l', 'o', 'w', '</w>')
    """
    return tuple(list(word) + ["</w>"])

def get_vocab(corpus: List[str]) -> Dict[Tuple, int]:
    """Build initial character-level vocabulary with frequencies.
    Returns: {char_tuple: count}
    """
    vocab = Counter()
    for word in corpus:
        vocab[word_to_chars(word)] += 1
    return dict(vocab)

def get_pair_counts(vocab: Dict[Tuple, int]) -> Counter:
    """Count all adjacent pairs across the vocab, weighted by word frequency.
    
    EXAMPLE:
      vocab = {('l','o','w','</w>'): 4}
      → pairs: ('l','o'): 4, ('o','w'): 4, ('w','</w>'): 4
    """
    pairs = Counter()
    # YOUR CODE HERE
    # Hint: iterate over vocab items; for each word (tuple) and freq,
    # iterate over zip(word[:-1], word[1:]) to get adjacent pairs
    raise NotImplementedError("Implement get_pair_counts")
    return pairs

def merge_vocab(vocab: Dict[Tuple, int], pair: Tuple[str, str]) -> Dict[Tuple, int]:
    """Merge all occurrences of `pair` in vocab into a single token.
    
    EXAMPLE:
      pair = ('e', 's')
      ('n','e','w','e','s','t','</w>') → ('n','e','w','es','t','</w>')
    """
    new_vocab = {}
    bigram = pair  # e.g. ('e', 's')
    merged = ''.join(pair)  # e.g. 'es'
    # YOUR CODE HERE
    # Hint: for each word in vocab, scan for consecutive tokens matching bigram
    # and replace them with merged. Be careful with overlapping matches.
    raise NotImplementedError("Implement merge_vocab")
    return new_vocab

def run_bpe(corpus: List[str], num_merges: int) -> Tuple[Dict, List]:
    """Run full BPE for num_merges steps.
    Returns: (final_vocab, list_of_merge_rules)
    """
    vocab = get_vocab(corpus)
    merges = []
    for i in range(num_merges):
        pairs = get_pair_counts(vocab)
        if not pairs:
            break
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(vocab, best_pair)
        merges.append(best_pair)
        print(f"  Merge {i+1:2d}: {best_pair[0]!r} + {best_pair[1]!r} → {''.join(best_pair)!r}  (freq={pairs[best_pair]})")
    return vocab, merges

print("Running BPE on Whitmore corpus (10 merges)...")
final_vocab, merge_rules = run_bpe(CORPUS, num_merges=10)

# ── CHECKPOINT 1-A ───────────────────────────────────────────────────────────
assert merge_rules[0] == ('e', 's'), \
    f"First merge should be ('e','s'), got {merge_rules[0]}"
assert merge_rules[1] == ('es', 't'), \
    f"Second merge should be ('es','t'), got {merge_rules[1]}"
print("\n✅ CHECKPOINT 1-A passed — BPE merges match Whitmore's worked example.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 1-B: Encode New Text Using Learned Merge Rules
# ─────────────────────────────────────────────────────────────────────────────

def encode_word(word: str, merges: List[Tuple[str, str]]) -> List[str]:
    """Apply learned BPE merges to a new word in order.
    
    Algorithm: start with characters, then apply each merge rule
    in the order they were learned (greedy left-to-right scan).
    """
    tokens = list(word) + ["</w>"]
    for pair in merges:
        # YOUR CODE HERE
        # Apply this merge rule to `tokens` (list of strings)
        # Scan left-to-right; merge first occurrence of pair
        # (real BPE merges ALL occurrences in one pass — do that)
        pass
    return tokens

# Test on Whitmore's examples
for word in ["low", "newest", "lower", "widest", "lowest"]:
    tokens = encode_word(word, merge_rules)
    print(f"  {word:10s} → {tokens}")

# ── CHECKPOINT 1-B ───────────────────────────────────────────────────────────
enc = encode_word("newest", merge_rules)
assert "est" in enc or ("es" in enc and "t" in enc) or "newest</w>" in ''.join(enc), \
    f"'newest' should be merged to use 'est', got {enc}"

# Tokenisation tax demo — compare English vs Luganda
print("\n📊 Tokenisation Tax Demo (Lecture 3 §2.2)")
english_sent = "I am telling you"
luganda_equiv = "Nkugambirira"  # Single Luganda word — same meaning
for text in [english_sent, luganda_equiv]:
    chars = sum(len(w) for w in text.split())
    # Rough token count: characters / avg_merge_length
    tokens_approx = len([c for c in text if c != ' '])
    print(f"  {text!r:25s}: {chars} chars")
print("\n✅ CHECKPOINT 1-B passed — encoder works.")

---

## MISSION 2 — Cross-Entropy Loss in JAX

**Lecture 3, Section 3.3.**

The loss function is the engine of all learning. You must implement it from scratch using only `jnp` operations — no `optax.softmax_cross_entropy`, no cheating.

**Key equations:**
$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$
$$\mathcal{L} = -\log p(y^*) = -\log \text{softmax}(z)_{y^*}$$

**Numerical stability:** Compute `softmax` using the log-sum-exp trick:
$$\log \sum_j e^{z_j} = m + \log \sum_j e^{z_j - m}, \quad m = \max(z)$$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 2-A: Numerically Stable Cross-Entropy
# ─────────────────────────────────────────────────────────────────────────────

def stable_log_softmax(logits: jnp.ndarray) -> jnp.ndarray:
    """Compute log(softmax(logits)) using the log-sum-exp trick.
    Shape: (vocab_size,) → (vocab_size,)
    
    IMPORTANT: Do NOT compute softmax first then log — catastrophic cancellation.
    Compute log-softmax directly: z_i - logsumexp(z)
    """
    # YOUR CODE HERE
    # 1. Find max for numerical stability
    # 2. Compute log(sum(exp(z - max))) + max  <- this is logsumexp
    # 3. Return logits - logsumexp
    raise NotImplementedError("Implement stable_log_softmax")

def cross_entropy_loss(logits: jnp.ndarray, target_idx: int) -> jnp.ndarray:
    """Scalar cross-entropy loss for a single prediction.
    logits: (vocab_size,) — raw unnormalised scores
    target_idx: int — index of the correct token
    Returns: scalar loss
    """
    # YOUR CODE HERE
    # Hint: log_probs = stable_log_softmax(logits)
    # Loss = -log_probs[target_idx]
    raise NotImplementedError("Implement cross_entropy_loss")

def batch_cross_entropy(logits: jnp.ndarray, targets: jnp.ndarray) -> jnp.ndarray:
    """Mean cross-entropy over a batch.
    logits: (batch, vocab_size)
    targets: (batch,) — integer indices
    Returns: scalar mean loss
    
    Use jax.vmap to vectorise — do NOT write a Python for-loop.
    """
    # YOUR CODE HERE
    # Hint: vmap cross_entropy_loss over the batch dimension,
    # but cross_entropy_loss takes a scalar target — vmap handles that.
    raise NotImplementedError("Implement batch_cross_entropy")

# ── CHECKPOINT 2-A ───────────────────────────────────────────────────────────
key = random.PRNGKey(0)
test_logits = random.normal(key, (50000,))  # vocab_size = 50k
test_target = 7432  # Whitmore's example token index

loss_val = cross_entropy_loss(test_logits, test_target)
assert loss_val.shape == (), f"Expected scalar, got shape {loss_val.shape}"
assert float(loss_val) > 0, "Cross-entropy loss must be positive"

# Sanity: perfect prediction → loss ≈ 0
perfect_logits = jnp.full((50000,), -1e9).at[test_target].set(1e9)
perfect_loss = cross_entropy_loss(perfect_logits, test_target)
assert float(perfect_loss) < 1e-3, f"Perfect prediction should have ~0 loss, got {perfect_loss:.4f}"

# Sanity: uniform → loss ≈ log(vocab_size)
uniform_logits = jnp.zeros(50000)
uniform_loss = cross_entropy_loss(uniform_logits, test_target)
expected_uniform = np.log(50000)
assert abs(float(uniform_loss) - expected_uniform) < 0.01, \
    f"Uniform logits → loss should be ≈{expected_uniform:.3f}, got {float(uniform_loss):.3f}"

print(f"Random logits loss  : {float(loss_val):.4f}")
print(f"Perfect logits loss : {float(perfect_loss):.6f}")
print(f"Uniform logits loss : {float(uniform_loss):.4f}  (expected ≈ {expected_uniform:.4f})")
print("\n✅ CHECKPOINT 2-A passed — cross-entropy is numerically correct.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 2-B: Perplexity — What It Is and What It Isn't
# ─────────────────────────────────────────────────────────────────────────────

def perplexity(mean_cross_entropy_loss: float) -> float:
    """Compute perplexity from mean cross-entropy loss.
    Perplexity = exp(mean CE loss)
    Interpretation: 'on average, choosing between N equally likely tokens'
    """
    # YOUR CODE HERE — one line
    raise NotImplementedError

# Whitmore's benchmarks from Lecture 3, Section 3.3:
benchmarks = {
    "Random (50k vocab)": np.log(50000),
    "GPT-2 English"     : 3.0,   # approximate CE loss
    "GPT-4 English"     : 2.3,   # approximate CE loss
    "Human written text": 1.7,   # approximate CE loss (Whitmore: PPL 5-8)
}

print("Perplexity Benchmark Table")
print(f"{'Model':<25} {'CE Loss':>10} {'Perplexity':>12} {'Interpretation'}")
print("-" * 70)
for name, ce in benchmarks.items():
    ppl = perplexity(ce)
    print(f"{name:<25} {ce:>10.3f} {ppl:>12.1f}  ← choosing between {ppl:.0f} tokens")

# ── CHECKPOINT 2-B ───────────────────────────────────────────────────────────
assert abs(perplexity(np.log(50000)) - 50000) < 1, \
    "Perplexity of uniform distribution over 50k tokens should be 50000"
assert perplexity(0.0) == 1.0, "Zero loss → perplexity 1 (perfect certainty)"
print("\n✅ CHECKPOINT 2-B passed.")

---

## MISSION 3 — Backpropagation Through a Minimal Language Model

**Lecture 3, Sections 4–5.**

Build a tiny (but correct) one-layer language model in JAX. The architecture:
```
token_ids → Embedding → LayerNorm → Linear → logits → CrossEntropy
```

You will use **`jax.grad`** to compute exact gradients — no manual backprop formulas required. But you must understand what is happening at each layer.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 3-A: Model Parameters and Forward Pass
# ─────────────────────────────────────────────────────────────────────────────

# Hyperparameters — small enough to run on CPU
VOCAB_SIZE = 256    # byte-level for simplicity
D_MODEL    = 64     # embedding dimension
SEQ_LEN    = 16     # context window

class ModelParams(NamedTuple):
    """All trainable parameters as JAX arrays."""
    embed   : jnp.ndarray  # (vocab_size, d_model)
    ln_scale: jnp.ndarray  # (d_model,)  — LayerNorm gain
    ln_bias : jnp.ndarray  # (d_model,)  — LayerNorm shift
    W_out   : jnp.ndarray  # (d_model, vocab_size) — output projection
    b_out   : jnp.ndarray  # (vocab_size,)

def init_params(key: jnp.ndarray) -> ModelParams:
    """Initialise model parameters.
    
    Use Xavier/Glorot init for embed and W_out:
      std = sqrt(2 / (fan_in + fan_out))
    LayerNorm: scale=ones, bias=zeros (standard init)
    """
    k1, k2, k3 = random.split(key, 3)
    # YOUR CODE HERE
    # Hint: embed std ≈ sqrt(1/d_model)
    #       W_out std ≈ sqrt(2/(d_model+vocab_size))
    raise NotImplementedError("Implement init_params")

def layer_norm(x: jnp.ndarray, scale: jnp.ndarray, bias: jnp.ndarray,
               eps: float = 1e-5) -> jnp.ndarray:
    """Layer normalisation over last dimension.
    x: (..., d_model)
    
    Formula: (x - mean) / sqrt(var + eps) * scale + bias
    Compute mean and var over the LAST axis only.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement layer_norm")

def forward(params: ModelParams, token_ids: jnp.ndarray) -> jnp.ndarray:
    """Forward pass for a single sequence.
    token_ids: (seq_len,) int32
    Returns: logits (seq_len, vocab_size)
    
    Pipeline:
      1. Embed: token_ids → (seq_len, d_model)  [use params.embed as lookup table]
      2. LayerNorm on each position
      3. Linear: (seq_len, d_model) @ W_out + b_out → (seq_len, vocab_size)
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement forward")

# ── CHECKPOINT 3-A ───────────────────────────────────────────────────────────
key, subkey = random.split(MASTER_KEY)
params = init_params(subkey)

assert params.embed.shape == (VOCAB_SIZE, D_MODEL), \
    f"embed shape wrong: {params.embed.shape}"
assert params.W_out.shape == (D_MODEL, VOCAB_SIZE), \
    f"W_out shape wrong: {params.W_out.shape}"

# Test forward pass
test_tokens = jnp.array([72, 101, 108, 108, 111, 32, 74, 65, 88, 33, 0, 0, 0, 0, 0, 0],
                         dtype=jnp.int32)  # "Hello JAX!" in ASCII
logits = forward(params, test_tokens)
assert logits.shape == (SEQ_LEN, VOCAB_SIZE), \
    f"logits shape wrong: {logits.shape}, expected ({SEQ_LEN}, {VOCAB_SIZE})"

# Logits should be finite
assert jnp.all(jnp.isfinite(logits)), "Logits contain NaN/Inf — check LayerNorm or init"

print(f"Params: {sum(p.size for p in params):,} total weights")
print(f"Logits shape: {logits.shape}")
print(f"Logits range: [{float(logits.min()):.3f}, {float(logits.max()):.3f}]")
print("\n✅ CHECKPOINT 3-A passed — forward pass runs clean.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 3-B: Loss Function Over a Sequence (Causal LM Objective)
# ─────────────────────────────────────────────────────────────────────────────

def sequence_loss(params: ModelParams, token_ids: jnp.ndarray) -> jnp.ndarray:
    """Compute causal LM loss over a sequence.
    
    CAUSAL LANGUAGE MODELLING: predict token t from tokens 0..t-1.
    So the targets are token_ids[1:], and the logits used are from positions [:-1].
    
    token_ids: (seq_len,)
    Returns: mean cross-entropy loss (scalar)
    
    Steps:
      1. Forward pass → logits (seq_len, vocab_size)
      2. Input logits  = logits[:-1]   shape: (seq_len-1, vocab_size)
      3. Target tokens = token_ids[1:] shape: (seq_len-1,)
      4. Mean CE loss over all (seq_len-1) positions
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement sequence_loss")

# ── CHECKPOINT 3-B ───────────────────────────────────────────────────────────
loss = sequence_loss(params, test_tokens)
assert loss.shape == (), f"Expected scalar loss, got {loss.shape}"
assert float(loss) > 0, "Loss must be positive"

# With random init, loss should be close to log(VOCAB_SIZE) = log(256) ≈ 5.55
expected_init_loss = np.log(VOCAB_SIZE)
assert abs(float(loss) - expected_init_loss) < 1.5, \
    f"Random-init loss {float(loss):.3f} far from expected {expected_init_loss:.3f} — check init"

print(f"Initial loss     : {float(loss):.4f}")
print(f"Expected ≈ log(256): {expected_init_loss:.4f}")
print(f"Initial perplexity : {np.exp(float(loss)):.1f} tokens")
print("\n✅ CHECKPOINT 3-B passed — sequence loss correct.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 3-C: Compute Gradients with jax.grad + Verify Gradient Flow
# ─────────────────────────────────────────────────────────────────────────────

# JAX differentiates through Python functions automatically
# grad takes a function and returns a new function that computes the gradient
# w.r.t. the FIRST argument (or argnums= for others)

loss_and_grad = jax.value_and_grad(sequence_loss)  # gradients w.r.t. params

loss_val, grads = loss_and_grad(params, test_tokens)

print("Gradient norms (should all be > 0 — gradient is flowing):")
for name, g in zip(["embed", "ln_scale", "ln_bias", "W_out", "b_out"],
                   [grads.embed, grads.ln_scale, grads.ln_bias, grads.W_out, grads.b_out]):
    norm = float(jnp.linalg.norm(g))
    print(f"  ∂L/∂{name:<10}: norm = {norm:.6f}")

# ── CHECKPOINT 3-C ───────────────────────────────────────────────────────────
for name, g in zip(["embed", "ln_scale", "ln_bias", "W_out", "b_out"],
                   [grads.embed, grads.ln_scale, grads.ln_bias, grads.W_out, grads.b_out]):
    assert jnp.all(jnp.isfinite(g)), f"Gradient for {name} contains NaN/Inf"
    assert float(jnp.linalg.norm(g)) > 0, \
        f"Gradient for {name} is zero — gradient is NOT flowing (check forward pass)"

# CRITICAL INSIGHT: embed gradient is SPARSE
# Only rows corresponding to tokens in the sequence get non-zero gradient
nonzero_embed_rows = int(jnp.sum(jnp.any(grads.embed != 0, axis=1)))
print(f"\nEmbed grad sparsity: {nonzero_embed_rows}/{VOCAB_SIZE} rows non-zero")
print(f"(Only the {SEQ_LEN} tokens in the sequence got gradients — correct!)")
assert nonzero_embed_rows <= SEQ_LEN, \
    "More embedding rows updated than sequence length — something is wrong"

print("\n✅ CHECKPOINT 3-C passed — gradients flow correctly.")

---

## MISSION 4 — The Adam Optimiser: Implement It, Then Prove It Works

**Lecture 3, Section 6.**

Adam = Adaptive Moment Estimation. It maintains:
- **m**: first moment (running mean of gradients) → direction
- **v**: second moment (running mean of squared gradients) → per-parameter scale

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 4-A: Implement Adam from Scratch (No optax)
# ─────────────────────────────────────────────────────────────────────────────

class AdamState(NamedTuple):
    """Adam optimiser state."""
    m    : ModelParams  # first moment (same structure as params)
    v    : ModelParams  # second moment
    step : int          # current step count

def adam_init(params: ModelParams) -> AdamState:
    """Initialise Adam state: moments = zeros, step = 0."""
    # YOUR CODE HERE
    # Use jax.tree_util.tree_map to create zero arrays with same shapes as params
    raise NotImplementedError("Implement adam_init")

def adam_step(
    params   : ModelParams,
    grads    : ModelParams,
    state    : AdamState,
    lr       : float = 1e-3,
    beta1    : float = 0.9,
    beta2    : float = 0.999,
    eps      : float = 1e-8,
) -> Tuple[ModelParams, AdamState]:
    """One Adam update step.
    
    Returns: (new_params, new_state)
    
    Use jax.tree_util.tree_map for all operations — this handles
    the nested NamedTuple structure without writing per-field code.
    """
    t = state.step + 1
    # YOUR CODE HERE
    # 1. Update first moment: m = beta1*m + (1-beta1)*g
    # 2. Update second moment: v = beta2*v + (1-beta2)*g^2
    # 3. Bias correction: m_hat = m / (1 - beta1^t)
    #                     v_hat = v / (1 - beta2^t)
    # 4. Update: params = params - lr * m_hat / (sqrt(v_hat) + eps)
    raise NotImplementedError("Implement adam_step")

# ── CHECKPOINT 4-A ───────────────────────────────────────────────────────────
opt_state = adam_init(params)

# Run 3 steps and verify params change
p0 = params
for step in range(3):
    loss_val, grads = loss_and_grad(params, test_tokens)
    params, opt_state = adam_step(params, grads, opt_state)

# Params should have changed
delta = jnp.linalg.norm(params.W_out - p0.W_out)
assert float(delta) > 0, "Params did not change after 3 Adam steps"

# Loss should have gone down
loss_after = sequence_loss(params, test_tokens)
assert float(loss_after) < float(sequence_loss(p0, test_tokens)), \
    f"Loss should decrease: {float(sequence_loss(p0, test_tokens)):.4f} → {float(loss_after):.4f}"

print(f"Param delta (W_out L2) : {float(delta):.6f}")
print(f"Loss before 3 steps    : {float(sequence_loss(p0, test_tokens)):.4f}")
print(f"Loss after  3 steps    : {float(loss_after):.4f}")
print("\n✅ CHECKPOINT 4-A passed — Adam updates work.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 4-B: JIT Compile the Training Step
# ─────────────────────────────────────────────────────────────────────────────
# This is where JAX earns its reputation.
# jit traces the Python function once and compiles it to XLA.
# Subsequent calls skip Python entirely — orders of magnitude faster.

def train_step(params, opt_state, token_ids, lr=1e-3):
    """One full training step: forward + grad + Adam update."""
    loss_val, grads = loss_and_grad(params, token_ids)
    new_params, new_state = adam_step(params, grads, opt_state, lr=lr)
    return new_params, new_state, loss_val

# JIT compile — first call traces; subsequent calls use compiled version
jit_train_step = jax.jit(train_step)

# Benchmark: eager vs JIT
# Re-init params
key, subkey = random.split(key)
params = init_params(subkey)
opt_state = adam_init(params)

# Warm-up JIT
params, opt_state, _ = jit_train_step(params, opt_state, test_tokens)

N_BENCH = 200
t0 = time.time()
for _ in range(N_BENCH):
    params, opt_state, loss_val = jit_train_step(params, opt_state, test_tokens)
jax.block_until_ready(loss_val)  # wait for async dispatch
t_jit = (time.time() - t0) / N_BENCH * 1000

t0 = time.time()
p_tmp, s_tmp = init_params(key), adam_init(init_params(key))
for _ in range(N_BENCH):
    p_tmp, s_tmp, l_tmp = train_step(p_tmp, s_tmp, test_tokens)
t_eager = (time.time() - t0) / N_BENCH * 1000

print(f"Eager step time : {t_eager:.3f} ms")
print(f"JIT   step time : {t_jit:.3f} ms")
print(f"Speedup         : {t_eager/t_jit:.1f}×")
print("\n✅ CHECKPOINT 4-B — JIT should be faster (typically 5–50× on CPU).")

---

## MISSION 5 — Full Training Loop on Character-Level Text

**Lecture 3, Sections 4–6, + Whitmore Misconceptions 1, 3.**

Train the model on real text. Plot training *and* validation loss to catch overfitting.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 5-A: Data Pipeline
# ─────────────────────────────────────────────────────────────────────────────

# Use a short in-memory corpus — Whitmore's lecture excerpt (char-level, byte tokens)
RAW_TEXT = """
We train these models on the sum of human writing.
Not the best of human writing. Not the most accurate.
The sum. Wikipedia and Reddit. Medical journals and conspiracy forums.
Legal scholarship and tabloid journalism. Philosophy and hate speech.
The model learns to predict all of it.
It does not distinguish truth from fiction, care from cruelty,
knowledge from prejudice. It learns the statistical regularities
of what humans write which is a complicated, contradictory, beautiful,
and deeply flawed corpus.
Every capability that impresses you emerged from that data.
Every failure that troubles you also emerged from that data.
They are not separable. You cannot remove the failures without removing the capabilities
because they share the same weights.
When you deploy these systems, you are deploying that corpus.
Know what you are deploying.
""" * 8  # repeat to get more data

def make_dataset(text: str, seq_len: int, train_frac: float = 0.9):
    """Convert text to overlapping token sequences (byte-level).
    Returns: (train_seqs, val_seqs) each shaped (N, seq_len)
    """
    # Encode to bytes (0-255)
    ids = jnp.array([b for b in text.encode('utf-8')], dtype=jnp.int32)
    # Create non-overlapping chunks of seq_len+1 (last token is target for last position)
    n_chunks = (len(ids) - 1) // seq_len
    seqs = [ids[i*seq_len : i*seq_len + seq_len] for i in range(n_chunks)]
    seqs = jnp.stack(seqs)  # (n_chunks, seq_len)
    
    split = int(len(seqs) * train_frac)
    return seqs[:split], seqs[split:]

train_seqs, val_seqs = make_dataset(RAW_TEXT, SEQ_LEN)
print(f"Training sequences   : {len(train_seqs)}  shape {train_seqs.shape}")
print(f"Validation sequences : {len(val_seqs)}  shape {val_seqs.shape}")
print(f"Total training tokens: {len(train_seqs) * SEQ_LEN:,}")
assert len(train_seqs) > 10, "Not enough training data — increase RAW_TEXT size"
print("\n✅ Dataset ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 5-B: Training Loop with Train/Val Loss Curves
# ─────────────────────────────────────────────────────────────────────────────

# YOUR TASK: implement the training loop below.
# Requirements:
#   - Use jit_train_step
#   - Track train loss per step
#   - Evaluate val loss every EVAL_EVERY steps (mean over all val sequences)
#   - Plot both curves at the end
#   - Detect and report if val loss diverges from train loss (overfitting indicator)

EPOCHS      = 3
LR          = 3e-3
EVAL_EVERY  = 10  # steps between val evals

key, subkey = random.split(key)
params      = init_params(subkey)
opt_state   = adam_init(params)

train_losses = []
val_losses   = []
val_steps    = []

global_step = 0
for epoch in range(EPOCHS):
    # Shuffle training data each epoch
    key, subkey = random.split(key)
    perm = random.permutation(subkey, len(train_seqs))
    shuffled = train_seqs[perm]
    
    for i, seq in enumerate(shuffled):
        # YOUR CODE HERE — training step + loss tracking
        raise NotImplementedError("Implement training loop body")
        
        global_step += 1
        
        if global_step % EVAL_EVERY == 0:
            # YOUR CODE HERE — compute val loss
            # Mean sequence_loss over all val_seqs
            raise NotImplementedError("Implement validation")
            
            print(f"  Step {global_step:4d} | Train loss: {train_losses[-1]:.4f} | "
                  f"Val loss: {val_losses[-1]:.4f}")

# ── CHECKPOINT 5-B ───────────────────────────────────────────────────────────
assert len(train_losses) > 0, "No training losses recorded"
assert train_losses[-1] < train_losses[0], \
    f"Training loss did not decrease: {train_losses[0]:.4f} → {train_losses[-1]:.4f}"
print(f"\nFinal train loss : {train_losses[-1]:.4f}")
print(f"Final val loss   : {val_losses[-1]:.4f}")
print(f"Loss reduction   : {train_losses[0] - train_losses[-1]:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 5-C: Plot Loss Curves + Whitmore Misconception Check
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Lecture 3 — Training Dynamics\n(Whitmore Misconception §10: training loss ≠ generalisation)",
             fontsize=13, fontweight='bold')

# Left: loss curves
ax = axes[0]
ax.plot(train_losses, alpha=0.6, label="Train loss", color='#2176AE')
ax.plot(val_steps, val_losses, 'o-', label="Val loss", color='#E84855', linewidth=2, markersize=4)
ax.set_xlabel("Training step")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
ax.set_title("Train vs Validation Loss")
ax.grid(alpha=0.3)

# Right: perplexity
ax = axes[1]
ax.plot([np.exp(l) for l in train_losses], alpha=0.6, label="Train PPL", color='#2176AE')
ax.plot(val_steps, [np.exp(l) for l in val_losses], 'o-', label="Val PPL",
        color='#E84855', linewidth=2, markersize=4)
ax.axhline(y=np.exp(np.log(VOCAB_SIZE)), color='gray', linestyle='--',
           label=f"Random baseline (PPL={VOCAB_SIZE})", alpha=0.6)
ax.set_xlabel("Training step")
ax.set_ylabel("Perplexity")
ax.legend()
ax.set_title("Perplexity (exp of loss)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/home/claude/loss_curves.png", dpi=120, bbox_inches='tight')
plt.show()

# Whitmore Misconception #1 diagnostic
final_train_ppl = np.exp(train_losses[-1])
final_val_ppl   = np.exp(val_losses[-1])
overfit_gap = final_val_ppl - final_train_ppl
print(f"\n📊 Whitmore Misconception #1 Check:")
print(f"  Train PPL  : {final_train_ppl:.2f}")
print(f"  Val PPL    : {final_val_ppl:.2f}")
print(f"  Gap        : {overfit_gap:.2f}")
if overfit_gap > 5:
    print("  ⚠️  OVERFITTING DETECTED — train loss is not generalisation")
else:
    print("  ✅ Generalisation gap acceptable for this corpus size")

---

## MISSION 6 — Chinchilla Scaling Law: The Calculator

**Lecture 3, Section 7.** 

The Hoffmann et al. 2022 result: *for every parameter, train on ~20 tokens.*

$$N_{\text{opt}} = \frac{C}{20}, \quad D_{\text{opt}} = 20 \cdot N$$

Where $C$ is compute budget in FLOPs and the factor of 20 comes from the measured optimal ratio. The full Chinchilla loss equation is:
$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$
with $E \approx 1.69$, $A \approx 406.4$, $B \approx 410.7$, $\alpha \approx 0.34$, $\beta \approx 0.28$.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 6-A: Implement the Chinchilla Loss Function and Scaling Calculator
# ─────────────────────────────────────────────────────────────────────────────

# Chinchilla paper constants (Hoffmann et al. 2022, Table 3)
CHINCHILLA = dict(E=1.693, A=406.4, B=410.7, alpha=0.3392, beta=0.2849)

def chinchilla_loss(N: float, D: float) -> float:
    """Predicted cross-entropy loss for a model with N parameters
    trained on D tokens, using Chinchilla fitted constants.
    
    L(N,D) = E + A/N^alpha + B/D^beta
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement chinchilla_loss")

def compute_optimal_allocation(compute_budget_flops: float) -> dict:
    """Given a compute budget C (in FLOPs), find optimal N and D.
    
    Training a transformer costs approximately 6*N*D FLOPs.
    So: C = 6*N*D
    
    Chinchilla optimal: N_opt = D_opt = sqrt(C / 6) / sqrt(20)
    (derived by setting D/N = 20 and substituting into C = 6*N*D)
    
    More precisely:
      D_opt = 20 * N_opt
      C = 6 * N_opt * D_opt = 6 * N_opt * 20 * N_opt = 120 * N_opt^2
      → N_opt = sqrt(C / 120)
    
    Returns dict with: N_opt, D_opt, predicted_loss
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement compute_optimal_allocation")

# Reproduce Whitmore's Table 7.2
models = [
    ("GPT-3",       175e9, 300e9,  "No — ~10× undertrained"),
    ("Gopher",      280e9, 300e9,  "No — ~18× undertrained"),
    ("Chinchilla",  70e9,  1.4e12, "Yes — ~20 tokens/param"),
    ("LLaMA-2-7B",  7e9,   2e12,   "Overtrained — intentional"),
    ("LLaMA-2-70B", 70e9,  2e12,   "Near-optimal"),
]

print(f"{'Model':<14} {'Params':>10} {'Tokens':>12} {'D/N ratio':>10} {'Predicted L':>12} {'Status'}")
print("-" * 80)
for name, N, D, status in models:
    ratio   = D / N
    pred_L  = chinchilla_loss(N, D)
    print(f"{name:<14} {N/1e9:>8.0f}B {D/1e9:>10.0f}B {ratio:>10.0f}× {pred_L:>12.3f}  {status}")

# ── CHECKPOINT 6-A ───────────────────────────────────────────────────────────
# GPT-3 undertrained → higher predicted loss than Chinchilla
L_gpt3  = chinchilla_loss(175e9, 300e9)
L_chinc = chinchilla_loss(70e9,  1.4e12)
assert L_gpt3 > L_chinc, \
    f"Chinchilla should predict lower loss than GPT-3 ({L_chinc:.3f} < {L_gpt3:.3f})"

# Optimal allocation for 1e21 FLOPs (roughly GPT-3 scale)
opt = compute_optimal_allocation(1e21)
assert 0.9e9 < opt['N_opt'] < 500e9, f"N_opt={opt['N_opt']:.2e} out of expected range"
assert abs(opt['D_opt'] / opt['N_opt'] - 20) < 1, "D/N ratio should be ≈ 20"

print(f"\nOptimal for 1e21 FLOP budget:")
print(f"  N_opt = {opt['N_opt']/1e9:.1f}B params")
print(f"  D_opt = {opt['D_opt']/1e9:.0f}B tokens")
print(f"  D/N   = {opt['D_opt']/opt['N_opt']:.1f}× (Chinchilla: ~20)")
print(f"  Pred L = {opt['predicted_loss']:.4f}")
print("\n✅ CHECKPOINT 6-A passed — Chinchilla calculator correct.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 6-B: Visualise the Chinchilla Loss Surface
# ─────────────────────────────────────────────────────────────────────────────

# YOUR TASK: produce a heatmap of predicted Chinchilla loss
# as a function of N (1B to 200B) and D (1B to 5T tokens).
# Overlay:
#   (1) The Chinchilla-optimal frontier: D = 20*N
#   (2) The GPT-3, Gopher, Chinchilla, LLaMA-2 points

N_range = np.logspace(9, 11.3, 60)   # 1B to 200B params
D_range = np.logspace(9, 12.7, 60)   # 1B to 5T tokens
NN, DD  = np.meshgrid(N_range, D_range)

# YOUR CODE HERE — compute loss surface
# loss_surface = chinchilla_loss(NN, DD)  <- should vectorise automatically
raise NotImplementedError("Compute and plot loss surface")

fig, ax = plt.subplots(figsize=(10, 7))

# Heatmap
# Overlay optimal frontier
# Annotate model points

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Parameters (N)')
ax.set_ylabel('Training tokens (D)')
ax.set_title('Chinchilla Loss Surface — Predicted CE Loss L(N, D)\nContour lines = iso-loss; diagonal = optimal (D = 20N)')
plt.tight_layout()
plt.savefig("/home/claude/chinchilla_surface.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Chinchilla surface plotted.")

---

## MISSION 7 — Gradient Clipping: The Stability Fix (Section 9.2)

Whitmore's debugging checklist from the Nairobi Kiswahili scenario includes:
> *"Add gradient clipping at norm 1.0. Transformer gradients can spike early in fine-tuning."*\n

Implement global gradient norm clipping and prove that it prevents loss explosions.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 7: Gradient Clipping + Stability Demo
# ─────────────────────────────────────────────────────────────────────────────

def global_grad_norm(grads: ModelParams) -> jnp.ndarray:
    """Compute global L2 norm across all gradient arrays.
    sqrt(sum of squared elements across ALL parameter tensors)
    """
    # YOUR CODE HERE
    # Use jax.tree_util.tree_leaves to get all gradient arrays
    raise NotImplementedError("Implement global_grad_norm")

def clip_grads(grads: ModelParams, max_norm: float = 1.0) -> ModelParams:
    """Clip gradients by global norm.
    If ||g|| > max_norm: g *= max_norm / ||g||
    If ||g|| <= max_norm: g unchanged
    
    This is the operation used in GPT-3, LLaMA, and virtually every
    production LLM training run.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement clip_grads")

# Stress test: inject a corrupted batch (simulate bad data)
# In real training, a corrupted batch causes gradient spikes
def corrupt_batch(key, tokens, corruption_prob=0.5):
    """Randomly replace tokens with extreme values to simulate data corruption."""
    mask = random.bernoulli(key, p=corruption_prob, shape=tokens.shape)
    corrupt = random.randint(key, tokens.shape, 0, VOCAB_SIZE)
    return jnp.where(mask, corrupt, tokens)

key, sk1, sk2 = random.split(key, 3)
corrupted_tokens = corrupt_batch(sk1, test_tokens)

_, grads_clean    = loss_and_grad(params, test_tokens)
_, grads_corrupt  = loss_and_grad(params, corrupted_tokens)

norm_clean   = global_grad_norm(grads_clean)
norm_corrupt = global_grad_norm(grads_corrupt)

grads_clipped = clip_grads(grads_corrupt, max_norm=1.0)
norm_clipped  = global_grad_norm(grads_clipped)

print("Gradient Norm Comparison:")
print(f"  Clean batch grad norm    : {float(norm_clean):.4f}")
print(f"  Corrupted batch grad norm: {float(norm_corrupt):.4f}")
print(f"  After clipping (max=1.0) : {float(norm_clipped):.4f}")

# ── CHECKPOINT 7 ─────────────────────────────────────────────────────────────
assert float(norm_clipped) <= 1.0 + 1e-5, \
    f"Clipped grad norm {float(norm_clipped):.4f} exceeds max_norm=1.0"
# Direction should be preserved (clipping only scales, does not change direction)
# Check by comparing dot product of clipped vs original
leaves_clip = jax.tree_util.tree_leaves(grads_clipped)
leaves_orig = jax.tree_util.tree_leaves(grads_corrupt)
dot = sum(float(jnp.sum(c * o)) for c, o in zip(leaves_clip, leaves_orig))
assert dot > 0, "Clipping reversed gradient direction — something is wrong"

print("\n✅ CHECKPOINT 7 passed — gradient clipping is correct.")
print(f"   Direction preserved (dot product = {dot:.2f} > 0 ✓)")

---

## MISSION 8 — BOSS LEVEL: Learning Rate Schedule (Cosine Decay + Warmup)

**Lecture 3, Section 6.3.** Every frontier training run uses this.

```python
if step < warmup_steps:
    lr = max_lr * step / warmup_steps
else:
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    lr = max_lr * 0.5 * (1 + cos(π * progress))
```

Implement, visualise the schedule, then train a model with and without warmup on a high learning rate (3e-2) and show that warmup prevents the early loss spike.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 8: Cosine LR Schedule + Warmup Ablation Study
# ─────────────────────────────────────────────────────────────────────────────

def cosine_lr_schedule(step: int, total_steps: int, max_lr: float,
                       warmup_steps: int = 0, min_lr: float = 0.0) -> float:
    """Cosine decay with optional linear warmup.
    Used verbatim in GPT-3 and most frontier models (Lecture 3, §6.3).
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement cosine_lr_schedule")

# ── Plot the schedule ────────────────────────────────────────────────────────
TOTAL_STEPS   = 500
WARMUP_STEPS  = 50
MAX_LR        = 3e-2

steps = np.arange(TOTAL_STEPS)
# YOUR CODE HERE — compute lr at each step

fig, ax = plt.subplots(figsize=(10, 4))
# YOUR CODE HERE — plot the schedule
ax.set_xlabel("Training step")
ax.set_ylabel("Learning rate")
ax.set_title(f"Cosine LR Schedule (max_lr={MAX_LR}, warmup={WARMUP_STEPS} steps)")
ax.axvline(WARMUP_STEPS, color='red', linestyle='--', alpha=0.5, label=f'Warmup ends (step {WARMUP_STEPS})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/home/claude/lr_schedule.png", dpi=120)
plt.show()

# ── Ablation: with warmup vs without warmup at high LR ───────────────────────
# Train two models: one with warmup, one without
# YOUR CODE HERE — run training loops for both configs
# Show that without warmup, the high LR causes an early loss spike or divergence
# This directly demonstrates Whitmore §9.2 debugging checklist item #2

# ── CHECKPOINT 8 ─────────────────────────────────────────────────────────────
# LR at step 0 with warmup should be very small
lr_step0 = cosine_lr_schedule(0, TOTAL_STEPS, MAX_LR, WARMUP_STEPS)
assert lr_step0 == 0.0 or lr_step0 < MAX_LR * 0.05, \
    f"LR at step 0 with warmup should be ~0, got {lr_step0:.4e}"

# LR at end should be near 0
lr_end = cosine_lr_schedule(TOTAL_STEPS - 1, TOTAL_STEPS, MAX_LR, WARMUP_STEPS)
assert lr_end < MAX_LR * 0.02, \
    f"LR at end should be near 0, got {lr_end:.4e}"

# LR at peak (warmup end) should equal max_lr
lr_peak = cosine_lr_schedule(WARMUP_STEPS, TOTAL_STEPS, MAX_LR, WARMUP_STEPS)
assert abs(lr_peak - MAX_LR) < 1e-8, \
    f"LR at warmup end should equal max_lr={MAX_LR}, got {lr_peak:.4e}"

print(f"LR at step 0           : {lr_step0:.4e}  (warmup start)")
print(f"LR at step {WARMUP_STEPS}          : {lr_peak:.4e}  (warmup end = max_lr)")
print(f"LR at step {TOTAL_STEPS-1}         : {lr_end:.4e}  (decay end)")
print("\n✅ CHECKPOINT 8 passed — LR schedule is correct.")

---

## MISSION 9 — OPEN FIELD: The Ugandan Medical Model Decision

**Lecture 3, Section 9.1 — Whitmore's Makerere Scenario**

> *"A research team at Makerere University has collected 40GB of Ugandan health text. Before they write a single line of training code, they need to answer three questions that the Chinchilla paper forces on them."*

This is the **ungraded, open-ended challenge**. There is no `assert`. There is no scaffolding. This is the real work.

### Your mission (choose one or tackle all three):

**9-A: The Tokenisation Tax Calculator**
Write a function that, given an English sentence and its Luganda equivalent (fetched from a translation API or hardcoded), measures the tokenisation tax — the ratio of token counts when encoded with a byte-level tokeniser vs a Luganda-aware BPE. Whitmore claims a 3× tax. Verify or challenge that claim.

**9-B: Train-from-Scratch vs Fine-Tune: A Compute Budget Argument**
Use your Chinchilla calculator (Mission 6) to make a quantitative argument for why fine-tuning is almost always correct for a 40GB corpus. Compute: (1) the optimal model size for 10B tokens from scratch, (2) what it would cost in FLOPs to train it, and (3) how much compute fine-tuning on a LLaMA-7B-sized model would cost for the same data. Output a table comparing both paths.

**9-C: The Curriculum-Aware Training Loop**
Implement a training loop that mixes two data distributions — general English text and domain-specific text — with a gradually increasing ratio of domain data over training. Start at 10% domain, end at 90%. Show that the final validation loss on domain data is lower than training on uniform mix. This is a simplified version of curriculum learning used in frontier medical LLMs.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MISSION 9 — YOUR IMPLEMENTATION HERE
# ─────────────────────────────────────────────────────────────────────────────

# No scaffolding. No CHECKPOINT. No partially-filled cells.
# This is the Wiseman-level problem: figure out what to build and build it.

# Hint for 9-A:
# Luganda test sentence: "Nkugambirira" (1 word = 'I am telling you')
# English equivalent: "I am telling you" (4 words, ~4 tokens in good BPE)
# Build a simple byte-level tokeniser and compare token counts

# Hint for 9-B:
# FLOPs for training ≈ 6 * N_params * D_tokens
# Compare: train_from_scratch(N_chinchilla_optimal, D=10e9)
# vs fine_tune(N=7e9, D=10e9, factor=0.01)  ← fine-tuning is ~1% of pretraining compute

# Hint for 9-C:
# general_corpus = RAW_TEXT (already loaded)
# domain_corpus  = """<some medical text you write or hardcode>"""
# At step t, sample domain text with probability = 0.1 + 0.8 * (t / total_steps)

print("Mission 9 is yours to fill. Think carefully before you code.")
print("Whitmore: 'Recitation is not understanding. Understanding is when you get stuck")
print(" on something nobody told you to get stuck on.'")

---

## Final Debrief — Whitmore's Seniority Questions Applied to This Notebook

Before you close this notebook, answer these in the cells below (prose, not code):

1. **What does your evaluation protocol not capture?** Your val loss measures next-character prediction on the same distribution. What failure modes of the model would this miss entirely?

2. **How was the training data collected and who was excluded?** Whitmore's corpus is English. What happens to this model's behaviour on non-ASCII characters? On Luganda text? Is the tokenisation fair?

3. **What happens to performance when the input distribution shifts?** Train on Whitmore's prose, then measure loss on Python source code. What do you expect? What does that tell you?

4. **What is the inference cost per token at scale, and does the performance gain justify it?** Your model is 256 × 64 × 256 ≈ 4M parameters. A LLaMA-7B is 7B. At 1,000 requests/second, what is the relative cost difference?

5. **How do we know when the model has forgotten something it used to know?** If you continue training this model on new data, how would you detect catastrophic forgetting?

---

*"You are not learning what a model does. You are learning what makes it possible."*  
*— Prof. Edmund R. Whitmore, MIT CSAIL, Final Academic Year 2025–2026*

In [ ]:
# ── Scorecard ─────────────────────────────────────────────────────────────────
missions = [
    ("1-A", "BPE pair counting + merging",      "§2 Byte Pair Encoding"),
    ("1-B", "BPE encoder + tokenisation tax",   "§1 Tokenisation Problem"),
    ("2-A", "Numerically stable cross-entropy", "§3.3 Loss Function"),
    ("2-B", "Perplexity + benchmarks",          "§3.3 Perplexity"),
    ("3-A", "Model init + forward pass",        "§4 Forward Pass"),
    ("3-B", "Causal LM sequence loss",          "§3.1 Training Objective"),
    ("3-C", "jax.grad + gradient flow check",   "§5 Backpropagation"),
    ("4-A", "Adam from scratch",                "§6 Adam Optimiser"),
    ("4-B", "jit_train_step speedup",           "§6 JIT Compilation"),
    ("5-A", "Data pipeline",                    "§4 Forward Pass (data)"),
    ("5-B", "Training loop + train/val loss",   "§10 Misconceptions #1"),
    ("6-A", "Chinchilla loss calculator",       "§7 Chinchilla Laws"),
    ("6-B", "Chinchilla loss surface plot",     "§7 Chinchilla Laws"),
    ("7",   "Gradient clipping",                "§9.2 Kiswahili Debug"),
    ("8",   "Cosine LR schedule + warmup",      "§6.3 LR Scheduling"),
    ("9",   "Open field: Makerere scenario",    "§9.1 Ugandan Medical LM"),
]

print("\n" + "="*75)
print("  MIT Masterclass Lecture 3 — JAX Field Implementation Scorecard")
print("="*75)
print(f"{'Mission':<8} {'Task':<40} {'Lecture Section'}")
print("-"*75)
for mid, task, section in missions:
    marker = "[ ]"  # replace with [✓] as you pass each checkpoint
    print(f"{marker} {mid:<6} {task:<40} {section}")
print("="*75)
print("\nMissions 1–8: gated by CHECKPOINT assertions (must pass to advance)")
print("Mission 9   : open field — judged by depth of reasoning, not correctness")